In [1]:
!pip install transformers==4.41.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [2]:
!pip install bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, EsmModel
import torch
import numpy as np
from tqdm.notebook import tqdm
from Bio import SeqIO
import gc
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t33_650M_UR50D")

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

model = model.to(device)
model.eval()

Using device: cuda
Available GPUs: 2


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using 2 GPUs!


DataParallel(
  (module): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 1280, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05,

In [4]:
def get_sequence_length_stats(sequences):
    lengths = [len(str(record.seq)) for record in sequences]
    return {
        'min': min(lengths),
        'max': max(lengths),
        'mean': np.mean(lengths),
        'median': np.median(lengths),
        'std': np.std(lengths)
    }

def smart_batching(sequences, batch_size, max_length_diff=200):
    seq_data = [(i, record, len(str(record.seq))) 
                for i, record in enumerate(sequences)]
    seq_data.sort(key=lambda x: x[2])
    
    batches = []
    current_batch = []
    
    for idx, record, length in seq_data:
        if not current_batch:
            current_batch.append((idx, record, length))
        else:
            min_len = min(item[2] for item in current_batch)
            max_len = max(item[2] for item in current_batch)
            
            if (len(current_batch) < batch_size and 
                max(max_len, length) - min(min_len, length) <= max_length_diff):
                current_batch.append((idx, record, length))
            else:
                batches.append(current_batch)
                current_batch = [(idx, record, length)]
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

def embed_fasta_esm2_optimized(fasta_path, output_prefix, batch_size=16, 
                               max_seq_length=1024, use_smart_batching=True,
                               save_every=5000, use_fp16=True):
    
    if Path(f"{output_prefix}_embeddings.npy").exists():
        print(f"Embeddings already exist at {output_prefix}_embeddings.npy")
        return
    
    ids, embeds = [], []
    
    sequences = list(SeqIO.parse(fasta_path, "fasta"))
    total_seqs = len(sequences)
    print(f"Total sequences in {fasta_path}: {total_seqs}")
    
    stats = get_sequence_length_stats(sequences)
    print(f"\nSequence length statistics:")
    print(f"  Min: {stats['min']}, Max: {stats['max']}")
    print(f"  Mean: {stats['mean']:.1f}, Median: {stats['median']:.1f}")
    print(f"  Std: {stats['std']:.1f}")
    
    if use_smart_batching:
        print("\nCreating smart batches by sequence length...")
        batches = smart_batching(sequences, batch_size)
        print(f"Created {len(batches)} smart batches")
    else:
        batches = []
        for i in range(0, total_seqs, batch_size):
            batch_data = [(j, sequences[j], len(str(sequences[j].seq))) 
                         for j in range(i, min(i + batch_size, total_seqs))]
            batches.append(batch_data)
    
    result_embeds = [None] * total_seqs
    result_ids = [None] * total_seqs
    
    if use_fp16 and torch.cuda.is_available():
        print("Using mixed precision (FP16) for faster computation")
    
    processed = 0
    for batch_idx, batch_data in enumerate(tqdm(batches, desc="ESM2 Embedding")):
        try:
            indices = [item[0] for item in batch_data]
            batch_records = [item[1] for item in batch_data]
            batch_seqs = [str(record.seq)[:max_seq_length] for record in batch_records]
            batch_ids = [record.id for record in batch_records]
            
            inputs = tokenizer(
                batch_seqs, 
                return_tensors='pt', 
                padding=True, 
                truncation=True, 
                max_length=max_seq_length,
                add_special_tokens=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                if use_fp16 and torch.cuda.is_available():
                    with torch.cuda.amp.autocast():
                        outputs = model(**inputs)
                        if hasattr(outputs, 'last_hidden_state'):
                            embeddings = outputs.last_hidden_state
                        else:
                            embeddings = outputs[0]
                else:
                    outputs = model(**inputs)
                    if hasattr(outputs, 'last_hidden_state'):
                        embeddings = outputs.last_hidden_state
                    else:
                        embeddings = outputs[0]
                
                attention_mask = inputs['attention_mask']
                mask_expanded = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
                sum_embeddings = torch.sum(embeddings * mask_expanded, dim=1)
                sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
                embeddings = (sum_embeddings / sum_mask).cpu().numpy()
                
                for idx, original_idx in enumerate(indices):
                    result_embeds[original_idx] = embeddings[idx]
                    result_ids[original_idx] = batch_ids[idx]
            
            processed += len(batch_data)
            
            if (processed % save_every == 0) or (batch_idx == len(batches) - 1):
                valid_embeds = [e for e in result_embeds if e is not None]
                valid_ids = [i for i in result_ids if i is not None]
                
                if valid_embeds:
                    np.save(f"{output_prefix}_embeddings_temp.npy", np.stack(valid_embeds))
                    np.save(f"{output_prefix}_ids_temp.npy", np.array(valid_ids))
                
                print(f"{processed}/{total_seqs} embeddings calculated | Checkpoint saved")
            
            if batch_idx % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\nOOM error at batch {batch_idx}. Clearing cache and retrying...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise e
    
    final_embeds = np.stack(result_embeds)
    final_ids = np.array(result_ids)
    
    np.save(f"{output_prefix}_embeddings.npy", final_embeds)
    np.save(f"{output_prefix}_ids.npy", final_ids)
    
    for temp_file in [f"{output_prefix}_embeddings_temp.npy", 
                      f"{output_prefix}_ids_temp.npy"]:
        if Path(temp_file).exists():
            Path(temp_file).unlink()
    
    print(f"\nFinished {output_prefix}: {len(final_embeds)} embeddings saved.")
    print(f"ESM2 embeddings shape: {final_embeds.shape}")
    print(f"Embedding dimension: {final_embeds.shape[1]}")
    
    return final_embeds, final_ids

def embed_fasta_chunked(fasta_path, output_prefix, chunk_size=10000, **kwargs):
    sequences = list(SeqIO.parse(fasta_path, "fasta"))
    total_seqs = len(sequences)
    num_chunks = (total_seqs + chunk_size - 1) // chunk_size
    
    print(f"Processing {total_seqs} sequences in {num_chunks} chunks")
    
    all_embeds, all_ids = [], []
    
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_seqs)
        
        print(f"\nProcessing chunk {chunk_idx + 1}/{num_chunks}")
        
        chunk_fasta = f"temp_chunk_{chunk_idx}.fasta"
        SeqIO.write(sequences[start_idx:end_idx], chunk_fasta, "fasta")
        
        chunk_embeds, chunk_ids = embed_fasta_esm2_optimized(
            chunk_fasta, 
            f"temp_chunk_{chunk_idx}",
            **kwargs
        )
        
        all_embeds.append(chunk_embeds)
        all_ids.extend(chunk_ids)
        
        Path(chunk_fasta).unlink()
        Path(f"temp_chunk_{chunk_idx}_embeddings.npy").unlink()
        Path(f"temp_chunk_{chunk_idx}_ids.npy").unlink()
    
    final_embeds = np.vstack(all_embeds)
    final_ids = np.array(all_ids)
    
    np.save(f"{output_prefix}_embeddings.npy", final_embeds)
    np.save(f"{output_prefix}_ids.npy", final_ids)
    
    print(f"\nAll chunks processed: {final_embeds.shape}")

embed_fasta_esm2_optimized(
    "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta", 
    "test", 
    batch_size=32,
    use_smart_batching=True,
    use_fp16=True,
    save_every=5000
)

embed_fasta_esm2_optimized(
    "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta", 
    "train", 
    batch_size=32,
    use_smart_batching=True,
    use_fp16=True,
    save_every=5000
)

Total sequences in /kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta: 224309

Sequence length statistics:
  Min: 2, Max: 35213
  Mean: 429.2, Median: 342.0
  Std: 442.1

Creating smart batches by sequence length...
Created 7036 smart batches
Using mixed precision (FP16) for faster computation


ESM2 Embedding:   0%|          | 0/7036 [00:00<?, ?it/s]

/tmp/ipykernel_19/4212828157.py:95: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
2025-10-27 12:46:22.436562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761569182.666978      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761569182.729310      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/tmp/ipykernel_19/4212828157.py:95: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


20000/224309 embeddings calculated | Checkpoint saved
40000/224309 embeddings calculated | Checkpoint saved
60000/224309 embeddings calculated | Checkpoint saved
80000/224309 embeddings calculated | Checkpoint saved
100000/224309 embeddings calculated | Checkpoint saved
120000/224309 embeddings calculated | Checkpoint saved
140000/224309 embeddings calculated | Checkpoint saved
160000/224309 embeddings calculated | Checkpoint saved
180000/224309 embeddings calculated | Checkpoint saved
200000/224309 embeddings calculated | Checkpoint saved
220000/224309 embeddings calculated | Checkpoint saved
224309/224309 embeddings calculated | Checkpoint saved

Finished test: 224309 embeddings saved.
ESM2 embeddings shape: (224309, 1280)
Embedding dimension: 1280
Total sequences in /kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta: 82404

Sequence length statistics:
  Min: 3, Max: 35213
  Mean: 525.8, Median: 409.0
  Std: 521.6

Creating smart batches by sequence length..

ESM2 Embedding:   0%|          | 0/2594 [00:00<?, ?it/s]

20000/82404 embeddings calculated | Checkpoint saved
40000/82404 embeddings calculated | Checkpoint saved
60000/82404 embeddings calculated | Checkpoint saved
80000/82404 embeddings calculated | Checkpoint saved
82404/82404 embeddings calculated | Checkpoint saved

Finished train: 82404 embeddings saved.
ESM2 embeddings shape: (82404, 1280)
Embedding dimension: 1280


(array([[ 0.11166933,  0.04282525,  0.08400351, ...,  0.08334437,
         -0.04517027,  0.0276518 ],
        [-0.00813222, -0.0094773 , -0.04671159, ..., -0.08256343,
          0.05426735,  0.07540083],
        [-0.00565355, -0.0643537 ,  0.01624952, ..., -0.03323893,
         -0.00625958, -0.04224967],
        ...,
        [ 0.05702669,  0.04583621,  0.09189898, ...,  0.0042143 ,
         -0.10218529, -0.009564  ],
        [ 0.04790429, -0.01115566,  0.04704031, ...,  0.04951618,
         -0.17740901,  0.1119229 ],
        [ 0.13494924,  0.01743619,  0.09089079, ...,  0.02208718,
         -0.17695048,  0.03128992]], dtype=float32),
 array(['sp|A0A0C5B5G6|MOTSC_HUMAN', 'sp|A0JNW5|BLT3B_HUMAN',
        'sp|A0JP26|POTB3_HUMAN', ..., 'sp|Q9Y7P7|YQ63_SCHPO',
        'sp|Q9Y7Q3|YQ6A_SCHPO', 'sp|Q9Y816|YOND_SCHPO'], dtype='<U25'))